In [1]:
import json
from pathlib import Path

def load_json(directory: str) -> list[dict]:
    directory = Path(directory)
    files = [f for f in directory.iterdir() if f.is_file()]
    files.sort()

    data = []
    for file in files:
        with open(file, 'r', encoding='utf-8') as f:
            content = f.read()
            data.append(json.loads(content))

    return data

human_data = load_json("../../datasets/cleaned/human")
ai_data = load_json("../../datasets/cleaned/ai")

print(f"{len(human_data)} human data found!")
print(f"{len(ai_data)} human data found!")

1500 human data found!
1500 human data found!


In [2]:
from sklearn.model_selection import train_test_split

human_train, human_test = train_test_split(human_data, train_size=0.8, random_state=42)
ai_train, ai_test = train_test_split(ai_data, train_size=0.8, random_state=42)

train_data = human_train + ai_train
test_data = human_test + ai_test

print(f"Get {len(train_data)} data for training and {len(test_data)} data for testing")

Get 2400 data for training and 600 data for testing


In [3]:
from transformers import AutoTokenizer

model_name = "microsoft/mdeberta-v3-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

def encode_document(item, max_length=512):
    text = item['content']

    encoding = tokenizer(text, truncation=True, max_length=max_length, return_offsets_mapping=True, padding=False)
    offsets = encoding['offset_mapping']

    sentence_token_indices = []
    sentence_labels = []

    for sentence in item['sentences']:
        start = sentence['start']
        end = sentence['end']

        token_indices = []

        for token_idx, (token_start, token_end) in enumerate(offsets):
            if token_start == token_end:
                continue

            if token_end > start and token_start < end:
                token_indices.append(token_idx)

        if len(token_indices) == 0:
            continue

        sentence_token_indices.append(token_indices)

        label = 1 if sentence['label'].lower() == "ai" else 0
        sentence_labels.append(label)

    encoding.pop('offset_mapping')
    encoding['sentence_token_indices'] = sentence_token_indices
    encoding['sentence_labels'] = sentence_labels

    return encoding

In [4]:
print(train_data[0])
encode_document(train_data[0])

{'text_id': '22909_4', 'document_id': '22909', 'content': 'Sedangkan dampak  negatifnya yaitu masalah pencemaran lingkungan  yang disebabkan oleh gas buang dari kendaraan bermotor. Selain kendaraan model baru terdapat berbagai merk dan tipe kendaraan lama yang masih dioperasikan.', 'type': 'content', 'lang': 'id', 'label': 'human', 'sentences': [{'start': 0, 'end': 120, 'label': 'human'}, {'start': 121, 'end': 220, 'label': 'human'}]}


{'input_ids': [1, 321, 61016, 332, 24134, 47165, 368, 648, 529, 4004, 15123, 92037, 90484, 616, 28295, 458, 302, 11881, 503, 260, 2146, 6550, 260, 94566, 1086, 35979, 35260, 694, 18392, 261, 260, 19976, 35979, 35260, 2947, 5552, 768, 13330, 694, 7015, 25371, 470, 260, 99271, 35979, 35260, 15275, 458, 260, 6886, 302, 34999, 503, 261, 2], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'sentence_token_indices': [[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28], [30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52]], 'sentence_labels': [0, 0]}

In [5]:
import torch
import torch.nn as nn
from transformers import AutoModel

class SentenceClassifier(nn.Module):
    def __init__(self, model_name=model_name, num_labels=2):
        super().__init__()

        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size

        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(hidden_size, num_labels, dtype=self.encoder.dtype)

    def forward(self, input_ids, attention_mask, sentence_token_indices, sentence_labels=None):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)

        hidden_states = outputs.last_hidden_state

        batch_sentence_logits = []
        batch_sentence_probs = []
        batch_losses = []

        for batch_idx in range(input_ids.size(0)):
            document_hidden = hidden_states[batch_idx]
            sentence_representations = []

            for token_indices in sentence_token_indices[batch_idx]:
                token_indices = torch.tensor(token_indices, device=document_hidden.device)

                sentence_hidden = document_hidden[token_indices]
                sentence_representation = sentence_hidden.mean(dim=0)

                sentence_representations.append(sentence_representation)

            if len(sentence_representations) == 0:
                continue

            sentence_representations = torch.stack(sentence_representations)
            sentence_representations = self.dropout(sentence_representations)

            logits = self.classifier(sentence_representations)
            probs = torch.softmax(logits, dim=-1)

            batch_sentence_logits.append(logits)
            batch_sentence_probs.append(probs)

            if sentence_labels is not None:
                labels = torch.tensor(sentence_labels[batch_idx], dtype=torch.long, device=document_hidden.device)
                loss_fn = nn.CrossEntropyLoss()

                loss = loss_fn(logits, labels)

                batch_losses.append(loss)

        if batch_losses:
            loss = torch.stack(batch_losses).mean()
        else:
            loss = None

        return {
            'loss': loss,
            'logits': batch_sentence_logits,
            'probs': batch_sentence_probs,
        }

In [6]:
from torch.utils.data import Dataset, DataLoader

class TextDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return encode_document(self.data[idx])

def collate_fn(batch):
    max_len = max(len(item['input_ids']) for item in batch)

    input_ids = []
    attention_mask = []
    sentence_token_indices = []
    sentence_labels = []

    for item in batch:
        pad_len = max_len - len(item['input_ids'])
        
        input_ids.append(item['input_ids'] + [tokenizer.pad_token_id] * pad_len)
        attention_mask.append(item['attention_mask'] + [0] * pad_len)
        
        sentence_token_indices.append(item['sentence_token_indices'])
        if 'sentence_labels' in item:
            sentence_labels.append(item['sentence_labels'])

    batch_dict = {
        'input_ids': torch.tensor(input_ids, dtype=torch.long),
        'attention_mask': torch.tensor(attention_mask, dtype=torch.long),
        'sentence_token_indices': sentence_token_indices
    }
    
    if sentence_labels:
        batch_dict['sentence_labels'] = sentence_labels

    return batch_dict

train_dataset = TextDataset(train_data)
test_dataset = TextDataset(test_data)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=True, collate_fn=collate_fn)

In [ ]:
from torch.optim import AdamW
from tqdm.auto import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

model = SentenceClassifier().to(device)
optimizer = AdamW(model.parameters(), lr=2e-5)

epochs = 3

for epoch in range(epochs):
    # Training
    model.train()
    total_train_loss = 0
    
    train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]")
    for batch in train_pbar:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        sentence_token_indices = batch['sentence_token_indices']
        sentence_labels = batch.get('sentence_labels')
        
        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask, sentence_token_indices, sentence_labels)
        
        loss = outputs['loss']
        if loss is not None:
            loss.backward()
            optimizer.step()
            total_train_loss += loss.item()
            train_pbar.set_postfix({'loss': f"{loss.item():.4f}"})
            
    avg_train_loss = total_train_loss / len(train_loader)
    
    # Evaluation
    model.eval()
    total_test_loss = 0
    correct_preds = 0
    total_preds = 0
    
    test_pbar = tqdm(test_loader, desc=f"Epoch {epoch+1}/{epochs} [Test]")
    with torch.no_grad():
        for batch in test_pbar:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            sentence_token_indices = batch['sentence_token_indices']
            sentence_labels = batch.get('sentence_labels')
            
            outputs = model(input_ids, attention_mask, sentence_token_indices, sentence_labels)
            
            loss = outputs['loss']
            if loss is not None:
                total_test_loss += loss.item()
                test_pbar.set_postfix({'loss': f"{loss.item():.4f}"})
            
            # Calculate accuracy
            if sentence_labels is not None:
                for doc_probs, doc_labels in zip(outputs['probs'], sentence_labels):
                    preds = torch.argmax(doc_probs, dim=-1).cpu().tolist()
                    for pred, label in zip(preds, doc_labels):
                        if pred == label:
                            correct_preds += 1
                        total_preds += 1
                    
    avg_test_loss = total_test_loss / len(test_loader)
    test_acc = correct_preds / total_preds if total_preds > 0 else 0
    
    print(f"Epoch {epoch+1}/{epochs} Summary:")
    print(f"Train Loss: {avg_train_loss:.4f} | Test Loss: {avg_test_loss:.4f} | Test Acc: {test_acc:.4f}\n")

Using device: cpu


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2Model LOAD REPORT from: microsoft/mdeberta-v3-base
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
mask_predictions.dense.weight              | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias            | UNEXPECTED |  | 
mask_predictions.classifier.bias           | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight    | UNEXPECTED |  | 
mask_predictions.classifier.weight         | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight          | UNEXPECTED |  | 
deberta.embeddings.word_embeddings._weight | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias      | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias          | UNEXPECTED |  | 
lm_predictions.lm_head.bias                | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight        | UNEXPECTED |  | 
mask_predictions.dense.bias                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from di

Epoch 1/3 [Train]:   0%|          | 0/600 [00:00<?, ?it/s]